# 03 - Comparaison des modeles

**Objectif** : Comparer les performances de tous les modeles (baselines + recents) et tirer des conclusions.

## Modeles compares
### Baselines
- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM
- TF-IDF + Naive Bayes
- BERT (bert-base-uncased)

### Modeles recents
- DistilBERT
- ModernBERT (dec 2024)
- NeoBERT (feb 2025)

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
METRICS_DIR = '../artifacts/metrics/'
FIGURES_DIR = '../artifacts/figures/'

## 1. Chargement des metriques

In [ ]:
import glob

all_results = []

# Load baselines
baselines_path = METRICS_DIR + 'baselines_metrics.json'
if os.path.exists(baselines_path):
    with open(baselines_path, 'r') as f:
        baselines = json.load(f)
        all_results.extend(baselines)

# Load transformers -- either from combined file or individual files
transformers_path = METRICS_DIR + 'transformers_metrics.json'
if os.path.exists(transformers_path):
    with open(transformers_path, 'r') as f:
        transformers = json.load(f)
        all_results.extend(transformers)

# Also load individual model metrics (from split notebooks 02a/02b/02c)
loaded_models = {r['model'] for r in all_results}
for path in sorted(glob.glob(METRICS_DIR + '*_metrics.json')):
    fname = os.path.basename(path)
    if fname in ('baselines_metrics.json', 'transformers_metrics.json'):
        continue
    with open(path, 'r') as f:
        data = json.load(f)
        # Handle both single dict and list format
        if isinstance(data, dict):
            data = [data]
        for entry in data:
            if entry['model'] not in loaded_models:
                all_results.append(entry)
                loaded_models.add(entry['model'])

df = pd.DataFrame(all_results)

# Ajout d'une colonne type
baseline_names = [
    'TF-IDF + Logistic Regression',
    'TF-IDF + Linear SVM',
    'TF-IDF + Naive Bayes',
    'BERT (bert-base-uncased)',
]
df['type'] = df['model'].apply(lambda x: 'Baseline' if x in baseline_names else 'Recent')

print(f"Nombre de modeles : {len(df)}")
print(df[['model', 'type', 'roc_auc_macro', 'f1_macro']].to_string(index=False))

## 2. Tableau comparatif global

In [ ]:
main_cols = ['model', 'type', 'roc_auc_macro', 'f1_macro', 'precision_macro',
             'recall_macro', 'hamming_loss', 'train_time_sec', 'inference_time_sec']

comparison = df[main_cols].sort_values('roc_auc_macro', ascending=False)

print("=" * 100)
print("COMPARAISON GLOBALE DES MODELES")
print("=" * 100)
print(comparison.to_string(index=False))

# Sauvegarder le tableau
comparison.to_csv(METRICS_DIR + 'comparaison_globale.csv', index=False)

## 3. Visualisation : ROC-AUC par modele

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#1f77b4' if t == 'Baseline' else '#e74c3c' for t in comparison['type']]
bars = ax.barh(comparison['model'], comparison['roc_auc_macro'], color=colors, edgecolor='white')

# Valeurs sur les barres
for bar, val in zip(bars, comparison['roc_auc_macro']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('ROC-AUC (macro)')
ax.set_title('ROC-AUC (macro) par modele')

# Legende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', label='Baseline'),
                   Patch(facecolor='#e74c3c', label='Recent')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'roc_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Visualisation : F1-score par modele

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

comp_f1 = df[['model', 'type', 'f1_macro']].sort_values('f1_macro', ascending=False)
colors = ['#1f77b4' if t == 'Baseline' else '#e74c3c' for t in comp_f1['type']]
bars = ax.barh(comp_f1['model'], comp_f1['f1_macro'], color=colors, edgecolor='white')

for bar, val in zip(bars, comp_f1['f1_macro']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('F1-score (macro)')
ax.set_title('F1-score (macro) par modele')
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analyse par label

In [ ]:
# ROC-AUC par label et par modele
roc_cols = [f'roc_auc_{label}' for label in LABEL_COLS]
roc_by_label = df[['model'] + roc_cols].set_index('model')
roc_by_label.columns = LABEL_COLS

fig, ax = plt.subplots(figsize=(14, 7))
roc_by_label.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('ROC-AUC par label et par modele')
ax.set_xlabel('Label')
ax.set_ylabel('ROC-AUC')
ax.legend(title='Modele', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(LABEL_COLS, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'roc_auc_by_label.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nROC-AUC par label :")
print(roc_by_label.round(4).to_string())

## 6. Analyse cout / performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Temps d'entrainement vs ROC-AUC
for _, row in df.iterrows():
    color = '#1f77b4' if row['type'] == 'Baseline' else '#e74c3c'
    axes[0].scatter(row['train_time_sec'], row['roc_auc_macro'],
                    color=color, s=100, zorder=5)
    axes[0].annotate(row['model'], (row['train_time_sec'], row['roc_auc_macro']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[0].set_xlabel('Temps d\'entrainement (s)')
axes[0].set_ylabel('ROC-AUC (macro)')
axes[0].set_title('Cout d\'entrainement vs Performance')
axes[0].legend(handles=legend_elements)

# Temps d'inference vs ROC-AUC
for _, row in df.iterrows():
    color = '#1f77b4' if row['type'] == 'Baseline' else '#e74c3c'
    axes[1].scatter(row['inference_time_sec'], row['roc_auc_macro'],
                    color=color, s=100, zorder=5)
    axes[1].annotate(row['model'], (row['inference_time_sec'], row['roc_auc_macro']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[1].set_xlabel('Temps d\'inference (s)')
axes[1].set_ylabel('ROC-AUC (macro)')
axes[1].set_title('Cout d\'inference vs Performance')
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'cost_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Heatmap de toutes les metriques

In [ ]:
heatmap_cols = ['roc_auc_macro', 'f1_macro', 'precision_macro', 'recall_macro', 'hamming_loss']
heatmap_data = df[['model'] + heatmap_cols].set_index('model')

# Inverser hamming_loss pour que "plus haut = meilleur" partout
heatmap_data['hamming_loss'] = 1 - heatmap_data['hamming_loss']
heatmap_data = heatmap_data.rename(columns={'hamming_loss': '1 - hamming_loss'})

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
            vmin=0.5, vmax=1.0, linewidths=0.5)
ax.set_title('Heatmap des metriques (plus haut = meilleur)')

plt.tight_layout()
plt.savefig(FIGURES_DIR + 'metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusion

In [ ]:
best_model = df.loc[df['roc_auc_macro'].idxmax()]
best_baseline = df[df['type'] == 'Baseline'].loc[df[df['type'] == 'Baseline']['roc_auc_macro'].idxmax()]
best_recent = df[df['type'] == 'Recent'].loc[df[df['type'] == 'Recent']['roc_auc_macro'].idxmax()]

print("=" * 70)
print("CONCLUSION")
print("=" * 70)
print(f"\nMeilleur modele global     : {best_model['model']} (ROC-AUC: {best_model['roc_auc_macro']:.4f})")
print(f"Meilleure baseline         : {best_baseline['model']} (ROC-AUC: {best_baseline['roc_auc_macro']:.4f})")
print(f"Meilleur modele recent     : {best_recent['model']} (ROC-AUC: {best_recent['roc_auc_macro']:.4f})")

gain = best_recent['roc_auc_macro'] - best_baseline['roc_auc_macro']
print(f"\nGain du meilleur recent vs meilleure baseline : {gain:+.4f} ROC-AUC")

print(f"\nTemps d'entrainement :")
print(f"  Meilleure baseline : {best_baseline['train_time_sec']:.1f}s")
print(f"  Meilleur recent    : {best_recent['train_time_sec']:.1f}s")
print(f"  Facteur            : x{best_recent['train_time_sec']/max(best_baseline['train_time_sec'], 0.01):.1f}")

In [ ]:
# Sauvegarder les metriques combinees pour le dashboard
with open(METRICS_DIR + 'all_metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nToutes les metriques sauvegardees dans {METRICS_DIR}all_metrics.json")
print("\nProchaine etape : dashboard Streamlit (dashboard/app.py)")